# Lab 03-03 — Qdrant: the in-memory cosine store

**Track 03 · Vector databases** — labs 01 and 02 covered the two ends of the persistence spectrum (pure RAM vs a directory on disk). Qdrant sits in the middle: one API, two local modes — `path=":memory:"` keeps everything in RAM, `path="<dir>"` persists to disk. This lab runs the in-memory mode and focuses on the second axis that separates vector stores: **what the score means**.

This notebook is **self-contained**: it imports LangChain, sentence-transformers, faiss, and qdrant directly — no repo component library. Every block of the pipeline is built right here: the BGE embedder, the FAISS baseline, and the in-memory Qdrant store all appear as plain code in the cells below, which is exactly how the shared components in `src/` work underneath.

The pipeline, drawn inline:

```
passages.parquet -> HuggingFaceEmbeddings (BGE, local) -> FAISS (sq-L2, lab-01 baseline)
                                                       -> Qdrant.from_documents(path=":memory:")
   -> similarity_search_with_score_by_vector -> top-3 per question
   -> cross-check: cos = 1 - sqL2/2 on every hit
```

FAISS (lab 01) and Chroma (lab 02) both default to the l2 space and report the same raw SQUARED L2 distance — lower is more similar. Qdrant's default is Cosine distance, so it reports a cosine SIMILARITY — higher is more similar. Same embeddings, same corpus, same ranking, but the numbers are not interchangeable.

Because every vector here is unit-norm (BGE normalizes), the two scores are exactly linked:

    sqL2 = |a - b|^2 = 2 - 2 * cos(a, b)   =>   cos = 1 - sqL2 / 2

The lab exploits that identity as a cross-check: for each hit it prints the FAISS squared-L2 score, the value `1 - sqL2/2`, and the score Qdrant actually reports. They agree to ~4 decimals — which proves both stores are computing the same thing and only disagreeing about how to display it.

The other lesson is structural: `:memory:` writes nothing to disk and forgets everything at process exit (run the lab twice — the second run builds a brand-new index from scratch). Persistent Qdrant is literally a one-argument change: `path="qdrant_storage"` instead of `path=":memory:"`, reusing lab 02's reopen story.


## Setup

One prerequisite must hold before this notebook will run:

- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/` (`passages.parquet` + `test.parquet`), already fetched by the repo's manifest-verified fetchers.

No repo imports are needed: everything this notebook uses comes from `langchain-core`, `langchain-huggingface`, `langchain-community`, `langchain-qdrant`, `sentence-transformers`, `faiss-cpu`, and `pandas`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
%pip install -q sentence-transformers langchain-huggingface langchain-community faiss-cpu qdrant-client langchain-qdrant pandas


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import os
import time
from pathlib import Path

import pandas as pd

# LangChain + sentence-transformers + faiss + qdrant — the only libraries
# this notebook needs. Nothing is imported from the repo's src/ component
# library.
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_core.embeddings import Embeddings  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402
from langchain_qdrant import Qdrant  # noqa: E402

# Several scratch notebooks may run in parallel on this machine; keep the
# BLAS thread pool small so embedding does not thrash memory.
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_PASSAGES = 100` takes the same deterministic head of the 3200-passage corpus as labs 01/02; `QUESTION_IDS = [1606, 1610]` are real questions whose answers live inside the subset; `TOP_K = 3` is the per-question hit list; `COSINE_TOL = 1e-3` is how tightly `cos = 1 - sqL2/2` must hold (float32 rounding); and `COLLECTION = "lab03"` names the in-memory collection. `PREVIEW` truncates the passage previews the demo prints next to each hit.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 100  # deterministic head of the 3200-passage corpus
QUESTION_IDS = [1606, 1610]  # real questions; answers live inside the subset
TOP_K = 3
PREVIEW = 62
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_DIM = 768
COSINE_TOL = 1e-3  # cos = 1 - sqL2/2 holds up to float32 rounding
COLLECTION = "lab03"


## 2. Load — corpus + questions (same helpers as labs 01/02)

`load_passages` reads the first `n` rows of `passages.parquet` and returns `(passage_texts, passage_ids)` — the ids are the parquet row indices. `load_questions` pulls the requested `test.parquet` rows as `(question_id, question_text)` pairs, `preview` flattens a passage onto one line for printing, and `expected_cosine` is the unit-norm identity `cos(a,b) = 1 - |a-b|^2 / 2` that the whole cross-check rests on.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — corpus + questions (same helpers as labs 01/02)
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


def expected_cosine(sq_l2: float) -> float:
    """cos(a,b) = 1 - |a-b|^2 / 2, exact for unit-norm vectors."""
    return 1.0 - sq_l2 / 2.0


## 3. Experiment — embed once, index into FAISS (sq-L2) and Qdrant (cosine), then cross-check the two score conventions on every hit

The whole pipeline is built inline. The embedder is `HuggingFaceEmbeddings` with the local BGE model (`normalize_embeddings=True`, which makes every vector unit-norm — the precondition for the `cos = 1 - sqL2/2` identity); we embed the 100 passages once and hand BOTH stores the same vectors through a tiny precomputed passthrough. FAISS is the sq-L2 baseline (`similarity_search_with_score_by_vector`); Qdrant is built with `Qdrant.from_documents(..., path=":memory:")` — its default distance is Cosine, so `similarity_search_with_score_by_vector` returns a cosine similarity, higher = more similar. For every hit the demo prints the FAISS score, the implied `1 - sqL2/2`, and the Qdrant score. This is the same mechanism the shared `src/vectordb/qdrant.py` class wraps.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — embed once, index into FAISS (sq-L2) and Qdrant (cosine),
#    then cross-check the two score conventions on every hit
# --------------------------------------------------------------------------
class _PrecomputedEmbeddings(Embeddings):
    """Hand the store precomputed vectors (looked up BY TEXT, not by order).

    FAISS calls embed_documents once with the full list; langchain-qdrant
    batches at 64. The text-keyed lookup keeps the embed step and the index
    step separately timed and stays correct under either batching.
    """

    def __init__(self, texts: list[str], embeddings: list[list[float]]):
        if len(texts) != len(embeddings):
            raise ValueError("texts and embeddings must be parallel lists")
        self._table: dict[str, list[float]] = dict(zip(texts, embeddings))

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        missing = [t for t in texts if t not in self._table]
        if missing:
            raise ValueError(f"{len(missing)} text(s) have no precomputed vector")
        return [self._table[t] for t in texts]

    def embed_query(self, text: str) -> list[float]:
        raise NotImplementedError("precomputed embeddings cannot embed queries")


def run_experiment() -> dict:
    passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, QUESTION_IDS)
    chunks = [
        Document(page_content=t, metadata={"id": pid})
        for t, pid in zip(passage_texts, passage_ids)
    ]

    # --- Embed the subset once; BOTH stores index the same vectors ----------
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        encode_kwargs={"normalize_embeddings": True},  # BGE needs cosine-normalized vectors
    )
    t0 = time.perf_counter()
    passage_vecs = embedder.embed_documents(passage_texts)
    embed_s = time.perf_counter() - t0
    query_vecs = [embedder.embed_query(q) for _, q in questions]

    # --- FAISS (sq-L2, lower = better) — the cross-check baseline -----------
    faiss_store = FAISS.from_documents(
        chunks, embedding=_PrecomputedEmbeddings(passage_texts, passage_vecs)
    )
    faiss_scored = [
        faiss_store.similarity_search_with_score_by_vector(q, k=TOP_K)
        for q in query_vecs
    ]

    # --- Qdrant (cosine, higher = better) — in-memory, nothing on disk ------
    t0 = time.perf_counter()
    qdrant_store = Qdrant.from_documents(
        chunks,
        embedding=_PrecomputedEmbeddings(passage_texts, passage_vecs),
        collection_name=COLLECTION,
        path=":memory:",
    )
    add_s = time.perf_counter() - t0
    qdrant_scored = [
        qdrant_store.similarity_search_with_score_by_vector(q, k=TOP_K)
        for q in query_vecs
    ]

    return {
        "passage_texts": passage_texts,
        "passage_ids": passage_ids,
        "questions": questions,
        "embed_s": embed_s,
        "add_s": add_s,
        "faiss_scored": faiss_scored,
        "qdrant_scored": qdrant_scored,
        "dim": len(passage_vecs[0]),
        "indexed": len(passage_vecs),
    }


## 4. Demo — print the artifact

`print_demo(exp)` prints the artifact from four angles: the corpus subset with the embedding timing; the Qdrant index build (in-memory, zero bytes on disk); the top-3 per question with three columns — the FAISS sq-L2 score, the `1 - sqL2/2` value it implies, and the score Qdrant actually reports (they should match); then a takeaway explaining that the store never changes WHAT ranks first, only the score scale and direction — and that `:memory:` is a scratchpad, one argument away from a persistent store.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 03 — Qdrant: the in-memory cosine store")
    print(f"{BGE_MODEL_NAME} | cosine distance | path=:memory: (nothing on disk)")
    print("=" * 66)

    print(f"\n[1] Corpus + embedding:")
    print(f"    {exp['indexed']} passages, dim {exp['dim']}, embedded in {exp['embed_s']:.2f}s")
    print(f"    same vectors indexed into FAISS (sq-L2) AND Qdrant (cosine)")

    print(f"\n[2] Qdrant index build (in-memory):")
    print(f"    {exp['indexed']} passages added in {exp['add_s']:.3f}s")
    print("    no persist_dir: path=\":memory:\" writes zero bytes and forgets")
    print("    everything at process exit (persistent mode = path='<dir>' instead)")

    print(f"\n[3] Top-{TOP_K} per question — sq-L2 (FAISS) vs cosine (Qdrant):")
    print("    col '1 - sqL2/2' is the cosine value FAISS's score implies;")
    print("    col 'qdrant' is what Qdrant actually reports. They should match.")
    for i, (qid, qtext) in enumerate(exp["questions"]):
        print(f'\n    Q[{qid}] "{qtext}"')
        for (fdoc, fscore), (cdoc, cscore) in zip(
            exp["faiss_scored"][i], exp["qdrant_scored"][i]
        ):
            conv = expected_cosine(fscore)
            match = "SAME" if fdoc.metadata["id"] == cdoc.metadata["id"] else "DIFF"
            print(f"      faiss {fscore:8.4f} | 1-sqL2/2 {conv:8.4f} | qdrant {cscore:8.4f} "
                  f"| [passage {cdoc.metadata['id']}] {preview(cdoc.page_content)}  {match}")

    print("\n[4] Takeaway")
    print("    The ranking is identical to labs 01/02 — the store never changes")
    print("    WHAT ranks first, only the score scale and direction. For unit-")
    print("    norm vectors cos = 1 - sqL2/2, so '0.4255 sq-L2' and '0.7873")
    print("    cosine' are the same retrieval. Never compare scores across")
    print("    stores (or across embedding models); compare rankings. And:")
    print("    :memory: is a scratchpad — the same one-line API turns it into")
    print("    a persistent store when you pass a directory path.")


## 5. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: embedding dimension 768, exactly `N_PASSAGES` passages indexed, Qdrant ranks the same passages as FAISS in the same order, cosine scores descend with rank, every Qdrant score matches `1 - sqL2/2` computed from the FAISS score within `COSINE_TOL`, and the two content checks (Q1610's top-1 names the Spanish founder of Montevideo; Q1606's top-1 mentions Montevideo). This is the same gate the CI-style `--verify` run applies; every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    checks.append(("embedding dimension is 768 (BGE base)", exp["dim"] == BGE_DIM))
    checks.append((f"exactly {N_PASSAGES} passages indexed", exp["indexed"] == N_PASSAGES))

    # Same embeddings => same ranking, whatever the store.
    all_same_rank = True
    for fhits, qhits in zip(exp["faiss_scored"], exp["qdrant_scored"]):
        for (fdoc, _), (qdoc, _) in zip(fhits, qhits):
            all_same_rank &= fdoc.metadata["id"] == qdoc.metadata["id"]
    checks.append(("Qdrant ranks the same passages as FAISS, in the same order", all_same_rank))

    # The signature assertion: Qdrant's cosine score equals 1 - sqL2/2 for
    # every hit, i.e. both stores are computing the same similarity.
    cos_consistent = True
    descending = True
    for fhits, qhits in zip(exp["faiss_scored"], exp["qdrant_scored"]):
        q_scores = [s for _, s in qhits]
        descending &= q_scores == sorted(q_scores, reverse=True)
        for (_, fscore), (_, cscore) in zip(fhits, qhits):
            cos_consistent &= abs(expected_cosine(fscore) - cscore) < COSINE_TOL
    checks.append(("cosine scores descend with rank (higher = more similar)", descending))
    checks.append(("every Qdrant score matches 1 - sqL2/2 from the FAISS score", cos_consistent))

    # Content checks (same as labs 01/02).
    q1610_top = exp["qdrant_scored"][1][0][0].page_content.lower()
    checks.append(("Q1610 top-1 names the Spanish founder of Montevideo", "spanish" in q1610_top))
    q1606_top = exp["qdrant_scored"][0][0][0].page_content.lower()
    checks.append(("Q1606 top-1 mentions Montevideo", "montevideo" in q1606_top))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A few minutes of embedding + index build on rag-mini-wikipedia — no downloads, no API calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The top-3 per question with the three score columns — FAISS sq-L2, the implied `1 - sqL2/2`, and Qdrant's actual cosine — plus the in-memory index build timing.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the parquet files are intact.


In [ ]:
verify_gate(exp)
